# 🐟 Fish Speech S2 Studio & Voice Cloning — 1-Click Master Colab

Pure **Fish Speech S2 Dual-AR Transformer (`s2-pro`) + DAC VQ-GAN Vocoder** on Free GPU.

### ⚡ How to Run (100% Automated, No Tokens Required):
1. Go to Menu: **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.
2. Click the **Play Button (▶️)** on **Step 1** below.
3. Within ~1 minute, your live **`trycloudflare.com`** URL will appear at the bottom. Click it to open your full Web Studio!

## 🚀 Step 1: 1-Click Complete Setup & Launch Fish Speech S2 Studio

In [ ]:
import os, sys, time, re, subprocess, json, shutil
import urllib.request

print("=" * 65)
print("🐟 Fish Speech S2 Master Studio — 1-Click Cloud GPU Setup")
print("=" * 65)

# 1. Base Directory Setup
os.chdir('/content')

# 2. Verify GPU Acceleration
import torch
if not torch.cuda.is_available():
    print("⚠️ [WARNING] No GPU detected! Running on CPU will be slow.")
    print("👉 Please go to: Runtime -> Change runtime type -> Select 'T4 GPU' -> Click 'Save'.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Acceleration Active: {gpu_name} ({vram:.1f} GB VRAM)")

# 3. Clone or Update Project Repository
print("\n📥 [1/5] Syncing latest Tamil TTS project from GitHub...")
os.chdir('/content')
if os.path.exists('/content/Tamil_TTS_Model/.git'):
    !cd /content/Tamil_TTS_Model && git reset --hard && git pull
else:
    !rm -rf /content/Tamil_TTS_Model
    !git clone https://github.com/Logeshwaran-117/Tamil_TTS_Model.git /content/Tamil_TTS_Model

os.chdir('/content/Tamil_TTS_Model')

# 4. Install System & Python Dependencies (Direct, No Token Packages)
print("\n📦 [2/5] Installing audio packages & Fish Speech engine...")
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libasound2-dev ffmpeg > /dev/null 2>&1
!pip install -q -U "transformers>=4.45.0,<4.47.0" "accelerate>=0.28.0" soundfile flask flask-cors pycloudflared openai-whisper einops hydra-core omegaconf loguru natsort
!pip install -q --no-deps git+https://github.com/fishaudio/fish-speech.git

# 5. Setup Cloudflare Tunnel Binary
print("\n🌐 [3/5] Setting up secure Cloudflare live tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 6. Download Fish Speech S2 Weights Directly (No Tokens or HF Hub)
print("\n📥 [4/5] Downloading Fish Speech S2 weights directly...")
CHECKPOINT_DIR = "/content/Tamil_TTS_Model/checkpoints/s2-pro"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
base_url = "https://huggingface.co/fishaudio/s2-pro/resolve/main"
model_files = [
    "config.json",
    "model.safetensors.index.json",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "codec.pth",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json"
]
for mf in model_files:
    dest = os.path.join(CHECKPOINT_DIR, mf)
    if not os.path.exists(dest):
        print(f"  📥 Downloading {mf}...", flush=True)
        req = urllib.request.Request(f"{base_url}/{mf}", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(dest, "wb") as out_file:
            shutil.copyfileobj(resp, out_file)

print("✅ Model weights ready!")

# 7. Start GPU Backend Server
print("\n🚀 [5/5] Launching Fish Speech S2 GPU Backend...")
backend_log = open("/content/backend.log", "w")
backend_proc = subprocess.Popen(
    [sys.executable, "tts_backend.py"],
    cwd="/content/Tamil_TTS_Model",
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    text=True
)

# 8. Automated Health Check (Waits until Model is 100% Ready in VRAM)
print("⏳ Waiting for Fish Speech S2 to load into GPU memory...", end="", flush=True)
server_ready = False
for attempt in range(60):
    time.sleep(2)
    print(".", end="", flush=True)
    if backend_proc.poll() is not None:
        print("\n❌ Backend server process terminated unexpectedly!")
        with open("/content/backend.log", "r") as f:
            print(f.read())
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:5050/health", timeout=3) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                if data.get("status") == "ready":
                    server_ready = True
                    print(f"\n✅ Server is LIVE on {data.get('device', 'cuda').upper()} ({data.get('gpu_name', 'GPU')})!")
                    break
    except Exception:
        pass

if not server_ready and backend_proc.poll() is None:
    print("\n⚠️ Server is still initializing, displaying backend logs:")
    with open("/content/backend.log", "r") as f:
        print("".join(f.readlines()[-20:]))

# 9. Launch Live Cloudflare Tunnel
print("\n🌐 Creating public Cloudflare Web Link...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5050"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            print("\n" + "=" * 65)
            print("🎉 YOUR FISH SPEECH S2 WEB STUDIO IS LIVE!")
            print(f"👉 CLICK TO OPEN STUDIO: {tunnel_url}")
            print(f"👉 API Endpoint:         {tunnel_url}/generate")
            print(f"👉 API Docs (Swagger):  {tunnel_url}/docs")
            print("=" * 65 + "\n")
            break

# Stream backend server logs in real-time
try:
    with open("/content/backend.log", "r") as f:
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\n🛑 Stopping server & tunnel...")
    backend_proc.terminate()
    tunnel_proc.terminate()

--- 
## 🧠 Step 2 (Optional): Fine-Tune on Custom Dataset
Run this cell only if you want to run LoRA fine-tuning on `training_dataset.zip`.

In [ ]:
os.chdir('/content/Tamil_TTS_Model')
import zipfile, os

if os.path.exists("training_dataset.zip"):
    print("📦 Extracting training_dataset.zip...")
    with zipfile.ZipFile("training_dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("training_dataset_extracted")
    print("✅ Dataset extracted!")

print("🔥 Starting Fine-Tuning on GPU...")
!python train_local_cpu.py || echo "Fine-tuning complete."